In [1]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-1]) + '/data/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [2]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent

In [3]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=150, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.7 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="instruction_db",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [17]:

instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v3.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'

importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm)
assistant.prepare_instructions(instructions_travel,'travel')
assistant.prepare_instructions(instructions_passport,'passport')
assistant.prepare_patterns(pat)
assistant.setup_rag(collection,[['qa','travel',qa_travel],['qa','passport',qa_passport],['nav',nav]])

In [ ]:
assistant.chat()

Hallo! Ik kan je helpen met 'een reis plannen' of 'een afspraak maken voor een paspoort'. Je kan 'vorige' zeggen als je een stap terug wilt doen, 'volgende' als je naar de volgende stap wilt. Zeg 'herhaal' als je wilt dat ik een stap herhaal. Je kan altijd een vraag stellen. Zeg nu 'reis' of 'paspoort'.


You:  reis


{'ids': [['a8a84213-283e-4eee-a913-6527d3e644c7', 'd4a0f818-be61-43b1-a346-133b5d7c5fd6', '26f69654-c945-4295-b531-a725e6264329']], 'embeddings': None, 'documents': [['Reis', 'Reis', 'Reis']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'step_context': 'all', 'action': 'travel', 'type': 'nav'}, {'action': 'travel', 'type': 'nav', 'step_context': 'all'}, {'step_context': 'all', 'type': 'nav', 'action': 'travel'}]], 'distances': [[-2.384185791015625e-07, -2.384185791015625e-07, -2.384185791015625e-07]]}
MATCH ['Reis', 'Reis', 'Reis'] , DISTANCE -2.384185791015625e-07 , CAT nav , DO travel
Agent: Ik ga je instrueren om een reis met het ov te plannen op 9292.nl. Stap 1: Waar begint je reis? Vul dat in bij 'Van'.


You:  vorige


{'ids': [['79cdf8e7-be07-460f-b468-2959b95cd250', 'f880ea26-588f-48f5-a425-4e4dcc46dbf8', '6ed1a1f0-0e0f-41c4-bc9b-b8f6352c227c']], 'embeddings': None, 'documents': [['Vorige', 'Vorige', 'Vorige']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'step_context': 'all', 'action': 'previous step', 'type': 'nav'}, {'step_context': 'all', 'action': 'previous step', 'type': 'nav'}, {'type': 'nav', 'step_context': 'all', 'action': 'previous step'}]], 'distances': [[-1.1920928955078125e-07, -1.1920928955078125e-07, -1.1920928955078125e-07]]}
MATCH ['Vorige', 'Vorige', 'Vorige'] , DISTANCE -1.1920928955078125e-07 , CAT nav , DO previous step
Agent: Je zit al bij stap 1, dus ik kan geen stap terug instrueren.Waar begint je reis? Vul dat in bij 'Van'.


...

{'ids': [['0192188e-50fb-4397-a23a-df95fc10f883', '35bad4a1-fcc6-469d-9ff3-086567ac208e', '378db2c5-bf0d-40fb-a723-09ed2e009858']], 'embeddings': None, 'documents': [['Volgende', 'Volgende', 'Volgende']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'action': 'next step', 'type': 'nav', 'step_context': 'all'}, {'action': 'next step', 'step_context': 'all', 'type': 'nav'}, {'action': 'next step', 'type': 'nav', 'step_context': 'all'}]], 'distances': [[-3.5762786865234375e-07, -3.5762786865234375e-07, -3.5762786865234375e-07]]}
MATCH ['Volgende', 'Volgende', 'Volgende'] , DISTANCE -3.5762786865234375e-07 , CAT nav , DO next step
Agent: Wanneer wil je vertrekken? Als dit op een later moment is klik dan op de knop 'vertrek'


You:  volgende


{'ids': [['a788329f-e29e-43fe-bb73-c17ae18fdb1f', '35bad4a1-fcc6-469d-9ff3-086567ac208e', '36e921d3-16ad-493a-965a-827157708e2d']], 'embeddings': None, 'documents': [['Volgende', 'Volgende', 'Volgende']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'step_context': 'all', 'action': 'next step', 'type': 'nav'}, {'type': 'nav', 'step_context': 'all', 'action': 'next step'}, {'step_context': 'all', 'action': 'next step', 'type': 'nav'}]], 'distances': [[-3.5762786865234375e-07, -3.5762786865234375e-07, -3.5762786865234375e-07]]}
MATCH ['Volgende', 'Volgende', 'Volgende'] , DISTANCE -3.5762786865234375e-07 , CAT nav , DO next step
Agent: Als je op een latere dag wil reizen, klik dan op de knop 'datum' en kies de dag.


You:  volgende


{'ids': [['35bad4a1-fcc6-469d-9ff3-086567ac208e', '2de7ece4-6da2-43c0-ac9e-8bdf502721b4', '36e921d3-16ad-493a-965a-827157708e2d']], 'embeddings': None, 'documents': [['Volgende', 'Volgende', 'Volgende']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'action': 'next step', 'type': 'nav', 'step_context': 'all'}, {'type': 'nav', 'step_context': 'all', 'action': 'next step'}, {'step_context': 'all', 'action': 'next step', 'type': 'nav'}]], 'distances': [[-3.5762786865234375e-07, -3.5762786865234375e-07, -3.5762786865234375e-07]]}
MATCH ['Volgende', 'Volgende', 'Volgende'] , DISTANCE -3.5762786865234375e-07 , CAT nav , DO next step
Agent: Klik op de knop 'tijd'. Kies hoe laat je wilt vertrekken.
